# Subject 2 · Learner profiles: clustering a questionnaire

**Course "Artificial Intelligence"** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Chapter 3** (unsupervised learning: k-means) · **Mini-project, subject 2** (clustering surveys) · **Duration:** about 1 h 30

**What you will do**
- generate a fictional questionnaire filled in by 120 learners of English (motivation, reading, anxiety, use of AI…), then standardize it;
- choose the number of groups with the elbow method and run the k-means algorithm;
- describe and name the learner profiles you obtain, then ask yourself how they could be used in class, and with what precautions.

**How to use this notebook**
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. You never have to write code. Change only the fields of the cells titled "TO CHANGE". In Colab, the menu "Runtime → Run all" runs the whole notebook at once.

Before each code cell, a question asks you to **predict** the result: write down your answer, run the cell, then compare with the section "What you should see".

> **Fictional data.** All the answers in this notebook are made up by a program, from a fixed seed. No real learner was surveyed: never quote these figures as survey results.

## Step 1 · Load the tools

This cell loads the course libraries: NumPy (calculation), pandas (tables), matplotlib (charts) and scikit-learn (machine learning). It also fixes the order of the group colors.

**Before you run it:** will this cell display a table, a chart, or only a short message?

In [ ]:
#@title Load the tools (libraries)
import numpy as np                     # calculation on series of numbers
import pandas as pd                    # data tables
import matplotlib.pyplot as plt        # charts
from matplotlib.colors import LinearSegmentedColormap   # color gradient
from sklearn.preprocessing import StandardScaler        # standardization
from sklearn.cluster import KMeans                      # k-means algorithm
from sklearn.metrics import silhouette_score            # silhouette score

# Colors and shapes of the groups, always in the same order
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SHAPES = ["o", "s", "^", "D", "v", "P", "X", "*"]
print("Tools loaded.")

**What you should see:** a single message, `Tools loaded.` Loading produces nothing else: it prepares the work.

**Question:** find in the cell the name of the tool that will build the groups. Which word from the course do you recognize in its name?

## Step 2 · The questionnaire and its hidden profiles

Our fictional questionnaire asks six closed questions to undergraduate students who are learning English. Here is its **variable dictionary**, the document that the mini-project asks you to write for your own data:

| Variable | Question asked | Coding of the answer |
|---|---|---|
| `motivation` | "I am motivated to learn English." | 1 (strongly disagree) to 5 (strongly agree) |
| `reading` | "How many hours a week do you read in English, outside class?" | number of hours |
| `anxiety` | "I feel anxious when I have to speak English in class." | 1 to 5 |
| `ai_use` | "I use an AI tool (ChatGPT, machine translation) to do my homework." | 1 (never) to 5 (always) |
| `speaking` | "I speak up in class." | 1 (never) to 5 (very often) |
| `screen` | "How many hours a week do you watch series or videos in English?" | number of hours |

The anxiety question is inspired by a real scale from language didactics, the *Foreign Language Classroom Anxiety Scale* by Horwitz, Horwitz and Cope (1986), which has 33 statements.

The next cell chooses the random **seed** (the same number always gives the same answers) and describes four **hidden profiles**, A, B, C and D: for each one, the number of learners and the average answer to each question. The program will then draw answers around these averages. We do not tell you yet what these profiles represent: the algorithm itself will never see them.

**Before you run it:** reading the averages, which profile looks like heavy readers? Which one looks like very anxious learners?

In [ ]:
#@title TO CHANGE: the random seed, then run (the four hidden profiles)
seed = 2026 #@param {type:"integer"}
rng = np.random.default_rng(seed)      # reproducible random generator
columns = ["motivation", "reading", "anxiety", "ai_use", "speaking", "screen"]
scales = ["motivation", "anxiety", "ai_use", "speaking"]   # answers from 1 to 5
hours = ["reading", "screen"]                              # hours per week
hidden_profiles = {  # profile: (number of learners, average answer to each question)
    "A": (30, [4.5, 7.0, 2.0, 1.8, 4.0, 5]),
    "B": (35, [3.8, 3.5, 4.4, 2.3, 1.6, 4]),
    "C": (35, [3.6, 1.5, 2.2, 4.2, 3.8, 12]),
    "D": (20, [1.7, 0.5, 3.2, 4.4, 1.8, 2]),
}
spread = [0.7, 1.5, 0.7, 0.7, 0.7, 3]       # usual distance around each average
print(pd.DataFrame({name: avg for name, (n, avg) in hidden_profiles.items()}, index=columns))

**What you should see:** a table with 6 rows (the questions) and 4 columns (the profiles A, B, C, D). Profile A reads 7 hours a week, profile B has the highest anxiety (4.4) and the lowest speaking score (1.6), profile C spends 12 hours watching videos, profile D has the lowest motivation (1.7).

**Question:** these averages were chosen by the author of the notebook. In a real survey, who sets the profiles? Can we even know whether there are any?

This cell randomly draws the 120 answers around the averages of the profiles, rounds them like on a real questionnaire (whole numbers from 1 to 5 for the scales, whole hours), shuffles the rows, then **puts aside** the hidden profile column: the algorithm will only work on the six answers.

**Before you run it:** how many rows and how many columns will the table of answers have?

In [ ]:
#@title Randomly draw the 120 fictional answers
pieces = []
for name, (size, averages) in hidden_profiles.items():
    block = pd.DataFrame(rng.normal(averages, spread, size=(size, 6)), columns=columns)
    block["hidden_profile"] = name
    pieces.append(block)
answers = pd.concat(pieces, ignore_index=True)
# Round like on a real questionnaire: whole numbers from 1 to 5, whole hours
answers[scales] = answers[scales].round().clip(1, 5)
answers[hours] = answers[hours].round().clip(0, None)
answers[columns] = answers[columns].astype(int)
answers = answers.sample(frac=1, random_state=seed).reset_index(drop=True)      # shuffle
hidden_profile = answers.pop("hidden_profile")   # put the hidden profile aside
print("Rows, columns:", answers.shape)
answers.head(8)

**What you should see:** `Rows, columns: (120, 6)`, then the first 8 learners. The first one (row 0) answered 4, 3, 4, 2, 2, 2: motivated, 3 hours of reading, rather anxious, little AI use, rarely speaks up, 2 hours of videos. The third one (row 2) reads 7 hours a week with an anxiety of 1.

**Question:** reading only these 8 rows, can you guess the hidden profile of each learner? What makes the task difficult?

## Step 3 · Explore the answers

Before any clever calculation, we look at the data: mean, standard deviation (the spread around the mean), minimum and maximum of each question, then one histogram per question.

**Before you run it:** which question will have the largest numbers? Which one will have the highest standard deviation?

In [ ]:
#@title Summarize and draw the answers to each question
summary = answers.describe().round(1)
print(summary.loc[["mean", "std", "min", "max"]])   # mean, standard deviation, min, max
answers.hist(figsize=(10, 6), bins=12, color=COLORS[0], edgecolor="white", grid=False)
plt.suptitle("Distribution of the answers (fictional data, 120 learners)")
plt.tight_layout()
plt.show()

**What you should see:** a small table (rows `mean`, `std`, `min`, `max`) and six histograms. The `screen` question has the largest numbers (from 0 to 17 hours, mean 6.7) and the highest standard deviation (4.2); `reading` goes from 0 to 12 hours (standard deviation 2.7); the four scales stay between 1 and 5, with a standard deviation of about 1.1 to 1.3.

**Question:** if we measured the "distance" between two learners with these raw numbers, which question would count the most? Is that fair?

## Step 4 · Standardize

K-means groups together the learners who are **close** to each other. So that each question weighs the same, we standardize it: we subtract the mean and divide by the standard deviation. A standardized answer tells how many standard deviations the learner is from the mean: 0 = average, +1 = clearly above, −1 = clearly below.

standardized value = (answer − mean) / standard deviation

**Before you run it:** the first learner reads 3 hours a week, while the mean is 3.4 hours. Will their standardized value for `reading` be positive or negative? Close to 0 or to 1?

In [ ]:
#@title Standardize: put all the questions on the same scale
X = StandardScaler().fit_transform(answers[columns])
X_table = pd.DataFrame(X, columns=columns).round(2)
print("Three learners BEFORE (raw answers):")
print(answers[columns].head(3))
print()
print("The same AFTER standardization:")
print(X_table.head(3))
print()
print("Standard deviation of each question after:", X_table.std(ddof=0).round(1).tolist())

**What you should see:** for the first learner, `reading` goes from 3 to −0.14 (slightly below average), `anxiety` from 4 to 0.81 and `screen` from 2 to −1.12 (clearly below). All the questions now have a standard deviation of 1.0. Check by hand: (3 − 3.38) / 2.72 ≈ −0.14.

**Question:** the learner in row 2 reads 7 hours a week. Why is their standardized value (1.33) much smaller than 7?

## Step 5 · How many groups? The elbow method

The k-means algorithm needs to be given k, the number of groups. To choose it, we try k = 1, 2, … 8 and note the **inertia**: the sum of the squared distances between each learner and the center of their group. It always decreases as k increases; we keep the value of k after which it hardly decreases any more (the "elbow" of the curve). We also compute the **silhouette score** (from −1 to 1): the higher it is, the better separated the groups are.

**Before you run it:** where do you expect to see the elbow? Will the silhouette score be highest for the same k?

In [ ]:
#@title Elbow method: try from k = 1 to k = 8 groups
results = []
for k_try in range(1, 9):
    km = KMeans(n_clusters=k_try, n_init=10, random_state=0).fit(X)
    # The silhouette does not exist for a single group
    silhouette = silhouette_score(X, km.labels_) if k_try > 1 else np.nan
    results.append({"k": k_try, "inertia": round(km.inertia_, 1), "silhouette": round(silhouette, 3)})
elbow = pd.DataFrame(results)
print(elbow.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(elbow["k"], elbow["inertia"], marker="o", color=COLORS[0], linewidth=2)
ax.set_xlabel("Number of groups k")
ax.set_ylabel("Inertia (spread within the groups)")
ax.set_title("Elbow method (fictional data)")
plt.show()

**What you should see:** the inertia goes from 720 (k = 1) to 502.1, then 348.3, then **246.5 for k = 4**; after that, it only drops by about twenty points for each added group (226.2, 207.9, 197.4, 187.1). The elbow is at k = 4, and the silhouette score is also highest for k = 4 (0.387, against 0.342 for k = 3 and 0.324 for k = 5).

**Question:** why not simply choose k = 8, which has the lowest inertia? What would the inertia be with 120 groups, one per learner?

## Step 6 · Run k-means

The algorithm places k centers, assigns each learner to the closest center, recomputes each center as the mean of its group, and starts again until nobody changes group any more. `start` fixes the randomness of the initial placement of the centers; `n_init=10` reruns the algorithm from ten starts and keeps the best solution.

**Before you run it:** with k = 4, will the four groups have the same size? Remember the sizes of the hidden profiles (30, 35, 35, 20).

In [ ]:
#@title TO CHANGE: the number of groups k, then run (k-means)
k = 4 #@param {type:"slider", min:2, max:8, step:1}
start = 0 #@param {type:"integer"}
model = KMeans(n_clusters=k, n_init=10, random_state=start)
answers["group"] = model.fit_predict(X)     # one group number per learner
print("Number of learners in each group:")
print(answers["group"].value_counts().sort_index().to_string())

**What you should see:** four groups numbered from 0 to 3, of different sizes: 32, 21, 32 and 35 learners (120 in total). The sizes look like those of the hidden profiles without being identical.

**Question:** is group 0 "better" or "first" compared with group 3? What do these numbers mean, in your opinion?

## Step 7 · Describe the profiles

The group numbers mean nothing in themselves: only the **average profile** of each group lets you describe it. This cell computes the mean of each question in each group, in the units of the questionnaire (1 to 5 scale, hours).

**Before you run it:** which group do you think will have the highest mean for `reading`?

In [ ]:
#@title Compute the average profile of each group
profiles = answers.groupby("group")[columns].mean().round(1)
profiles["size"] = answers["group"].value_counts().sort_index()
profiles

**What you should see:** a table with 4 rows.
- Group 0 (32): motivation 4.4, reading 6.7 h, anxiety 2.1, AI 1.9, speaking 4.0, screen 6.3 h.
- Group 1 (21): motivation 1.6, reading 1.4 h, anxiety 3.3, AI 4.0, speaking 2.0, screen 2.6 h.
- Group 2 (32): motivation 3.5, reading 1.3 h, anxiety 2.6, AI 4.4, speaking 3.9, screen 11.6 h.
- Group 3 (35): motivation 3.9, reading 3.5 h, anxiety 4.3, AI 2.2, speaking 1.7, screen 4.9 h.

**Question:** groups 1 and 2 read as little as each other. On which questions are they clearly different?

A table of figures is hard to read. This cell draws a **portrait** of the groups: each cell gives the distance between the center of the group and the mean of all learners, in standard deviations (the standardized values of step 4). Red: above average; blue: below; gray: average.

**Before you run it:** which cell will be the reddest? Which one will be the bluest?

In [ ]:
#@title Draw the portrait of the groups (distance from the mean)
centers = pd.DataFrame(model.cluster_centers_, columns=columns)     # standardized centers
blue_red = LinearSegmentedColormap.from_list("gap", ["#2a78d6", "#f0efec", "#e34948"])
fig, ax = plt.subplots(figsize=(8, 1 + 0.7 * k))
image = ax.imshow(centers, cmap=blue_red, vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(columns)), columns)
ax.set_yticks(range(k), [f"Group {g}" for g in range(k)])
for g in range(k):
    for j in range(len(columns)):
        ax.text(j, g, f"{centers.iloc[g, j]:+.1f}", ha="center", va="center")
fig.colorbar(image, label="distance from the mean (in standard deviations)")
ax.set_title("Portrait of the groups: red = above the mean, blue = below")
plt.show()

**What you should see:** a grid with 4 rows and 6 columns. The reddest cells: `screen` for group 2 (+1.2) and `reading` for group 0 (+1.2). The bluest: `motivation` for group 1 (−1.7), then `screen` for group 1 and `speaking` for group 3 (−1.0). Group 3 also has an anxiety clearly above average (+1.1).

**Question:** read the row of group 1 again. Which cell seems the most important for a teacher, and why?

This cell turns the portrait into words: for each group, it lists the questions where the center of the group is further from the mean than a given **threshold** (in standard deviations). You can change this threshold with the slider.

**Before you run it:** with a threshold of 0.5, how many questions will be listed for group 0?

In [ ]:
#@title TO CHANGE: the threshold, then run (groups described in words)
threshold = 0.5 #@param {type:"slider", min:0.25, max:1.5, step:0.25}
for g in range(k):
    center = centers.iloc[g]
    above = [q for q in columns if center[q] > threshold]
    below = [q for q in columns if center[q] < -threshold]
    print(f"Group {g} ({profiles.loc[g, 'size']} learners)")
    print("   clearly above the mean:", ", ".join(above) or "nothing")
    print("   clearly below the mean:", ", ".join(below) or "nothing")

**What you should see:**
- Group 0 (32 learners): above the mean: motivation, reading, speaking; below: anxiety, ai_use.
- Group 1 (21): above: ai_use; below: motivation, reading, speaking, screen.
- Group 2 (32): above: ai_use, speaking, screen; below: reading.
- Group 3 (35): above: anxiety; below: ai_use, speaking.

**Question:** raise the threshold to 1.0 and run the cell again. What is left for each group? Is what remains the core of the profile?

## Step 8 · Name the profiles

It is now up to the researcher, that is you, to **name** the groups. A good name sums up the profile without turning it into a caricature. The names suggested below match the groups obtained with the default settings (seed 2026, k = 4, start 0); if you changed a setting, read the profiles again and rename them.

**Before you run it:** write down on paper your own name for each of the four groups, then compare with the names suggested in the fields. Replace them with yours if you prefer.

In [ ]:
#@title TO CHANGE: give a name to each group, then run
name_group_0 = "Independent readers" #@param {type:"string"}
name_group_1 = "Withdrawn learners" #@param {type:"string"}
name_group_2 = "Connected (screens, AI, speaking)" #@param {type:"string"}
name_group_3 = "Motivated but anxious" #@param {type:"string"}
names = {0: name_group_0, 1: name_group_1, 2: name_group_2, 3: name_group_3}
# A group without a name (if k is larger than 4) keeps its number
answers["profile"] = [names.get(g, f"Group {g}") for g in answers["group"]]
print(answers["profile"].value_counts().to_string())

**What you should see:** the four names with their sizes, from the largest to the smallest: Motivated but anxious 35, Independent readers 32, Connected (screens, AI, speaking) 32, Withdrawn learners 21.

Naming is interpreting. Like Max Weber's ideal types, these profiles are constructions that simplify reality to make it readable. A harsh name ("lazy", "AI cheaters") sticks to real people and can shape the teacher's view.

**Question:** does the name "Withdrawn learners" describe what these learners do or what they are? Suggest a more careful name, or on the contrary a more precise one.

## Step 9 · See the groups on a chart

Each learner has six answers: we cannot draw six dimensions. So we choose **two questions** and place each learner on a plane, with the color and shape of their group. A small random shift prevents identical points from hiding each other.

**Before you run it:** with `reading` on the horizontal axis and `ai_use` on the vertical axis, will the "Independent readers" be at the top left or at the bottom right?

In [ ]:
#@title TO CHANGE: two questions to compare, then run (scatter plot)
x_axis = "reading" #@param ["motivation", "reading", "anxiety", "ai_use", "speaking", "screen"] {allow-input: true}
y_axis = "ai_use" #@param ["motivation", "reading", "anxiety", "ai_use", "speaking", "screen"] {allow-input: true}
fig, ax = plt.subplots(figsize=(7, 5))
for g, group in answers.groupby("group"):
    shift = rng.uniform(-0.25, 0.25, size=(len(group), 2))      # small random shift
    ax.scatter(group[x_axis] + shift[:, 0], group[y_axis] + shift[:, 1], s=40, alpha=0.8,
               color=COLORS[g], marker=SHAPES[g], label=group["profile"].iloc[0])
ax.set_xlabel(x_axis)
ax.set_ylabel(y_axis)
ax.set_title("Each point is a fictional learner")
ax.legend(title="Profile")
plt.show()

**What you should see:** 120 points. The "Independent readers" (blue circles) are at the bottom right: a lot of reading, little AI. The "Motivated but anxious" (yellow diamonds) are in the lower middle: 1 to 6 hours of reading, little AI. The "Withdrawn learners" (orange squares) and the "Connected" (green triangles) mix at the top left: little reading, a lot of AI.

**Question:** choose two other questions that separate the "Withdrawn learners" well from the "Connected" (hint: look at the portrait in step 7), then run the cell again. Which ones did you choose?

## Step 10 · Reveal: did the algorithm find the hidden profiles?

Since our data are fictional, we know the "right answer": the hidden profile of each learner, put aside in step 2. This table crosses your groups (rows) with the hidden profiles (columns). Each cell counts the learners concerned.

**Before you run it:** out of 120, how many learners do you think will be in a group that does not match their hidden profile?

In [ ]:
#@title Compare the groups found with the hidden profiles
pd.crosstab(answers["profile"], hidden_profile, colnames=["hidden profile"])

**What you should see:** almost everything is in a single cell per row. Independent readers: 30 A and 2 C; Withdrawn learners: 20 D and 1 B; Connected: 32 C; Motivated but anxious: 34 B and 1 C. Only 4 learners out of 120 are "misplaced".

Here is what the author of the notebook had meant to build: **A** = heavy, independent readers, **B** = motivated learners who are anxious about speaking, **C** = learners who turn to screens and AI, and who speak up readily, **D** = demotivated learners.

**Question:** in a real survey, this "hidden profile" column does not exist. How could you then check that your groups make sense (reread questionnaires, interview learners, rerun with another start…)?

## Your turn

Change only the fields, then run the cells again from the one you changed (in Colab: "Runtime → Run after").
1. **Step 6:** set k = 3, then k = 5. Which profiles merge with k = 3? Which profile splits in two with k = 5? (Remember to reread step 7: the names of step 8 no longer match.)
2. **Step 6, stability:** set k = 4 again and change `start` (1, then 2). Do the group numbers change? Are the profiles still recognizable?
3. **Step 2:** change the seed (for example 7) and run the whole notebook again. Is the elbow still at k = 4?

## For the mini-project: your own data

For your mini-project, replace the fictional answers with those of **your** questionnaire:
- prepare a CSV file (in a spreadsheet: "Save as… → CSV") with **one row per respondent** and **one column per question**, each answer coded as a number;
- an anonymous questionnaire, filled in with the respondents' consent; remove names and identifiers (in Algeria, law no. 18-07 of 10 June 2018 regulates personal data);
- tick the box below and run the cell: in Colab, a window asks you for the file; in Jupyter, put the file next to the notebook under the name `my_questionnaire.csv`;
- the cell keeps the numeric columns and removes incomplete questionnaires (say so in your report); then run steps 4 to 9 again, typing the names of two of your questions in step 9. Skip step 10: your data have no hidden profile.

**Before you run it:** if you leave the box unticked, what will the cell do?

In [ ]:
#@title TO CHANGE: load your own questionnaire (CSV file)
load_my_file = False #@param {type:"boolean"}
if load_my_file:
    try:
        from google.colab import files       # only exists in Colab
        upload = files.upload()              # window to choose the file
        file_name = list(upload.keys())[0]
    except ImportError:                      # Jupyter: file placed next to the notebook
        file_name = "my_questionnaire.csv"
    my_answers = pd.read_csv(file_name)
    columns = list(my_answers.select_dtypes("number").columns)      # numeric questions
    answers = my_answers[columns].dropna().reset_index(drop=True)
    print(len(answers), "complete answers; questions:", columns)
else:
    print("Box not ticked: the notebook keeps the fictional data.")

**What you should see:** with the box unticked, `Box not ticked: the notebook keeps the fictional data.` With your file, the number of complete answers and the list of your numeric questions.

**Question:** can an open question ("Why are you learning English?") go into k-means as it is? How could you code it as numbers?

## To hand in

Answer in writing, in English, in a few lines for each question.
1. **Choosing k.** Justify the number of groups you kept using the elbow curve and the silhouette score. What do you lose with k = 3? What do you gain (or not) with k = 5?
2. **Describing and naming.** Describe the four profiles with the figures of the table in step 7, then justify the name you give to each one. Which name did you change, and why?
3. **In class (didactics).** For two of the profiles, suggest a suitable activity or teaching arrangement (differentiated instruction). Then state one limitation of this classification: sample size, self-reported answers, the risk of locking a learner into a label…

## Key points

- **Clustering** forms groups without labels given in advance: it is **unsupervised** learning. Before k-means, we **standardize**, otherwise the question with the largest numbers decides alone.
- The **elbow method** (inertia) and the **silhouette score** help to choose k, but the researcher decides, then describes the groups by their means.
- Group numbers are arbitrary; **profiles** are constructions, to be named with caution and checked against the field.

**Lessons on the course site:** [Mini-project, subject 2: clustering surveys](https://progremer04.github.io/canva-au-for-ang-/en/#mp-sujet-2) · [Chapter 3: clustering (k-means)](https://progremer04.github.io/canva-au-for-ang-/en/#c3-algorithmes)